# MP38: integrated hybrid reconstruction

Trace the existing shared model rather than creating a second implementation. Features are NCHW float32. These synthetic checks are separate from the unfinished matched-budget research comparison. See [lesson](README.md) for equations, diagram and exercises.

In [ ]:
import torch
from mp38_tools import build_hybrid,trace_hybrid
torch.manual_seed(38)
torch.set_num_threads(2)
x=torch.linspace(0,1,4*5*7).reshape(1,4,5,7)
model=build_hybrid(zero_residual=True)
y,shapes=trace_hybrid(model,x)
for name,shape in shapes.items():print(name,shape)
print("output",tuple(y.shape))

## Residual reconstruction

Output equals bicubic plus a gated residual. Zero tail implies zero residual, so the output equals bicubic exactly; this is an implementation check, not learned improvement. The sigmoid gate is not calibrated confidence.

In [ ]:
base=torch.nn.functional.interpolate(x,scale_factor=2,mode="bicubic",align_corners=False)
torch.testing.assert_close(y,base,atol=0,rtol=0)
model(x).sum().backward()
print("max difference",float((y-base).abs().max()))
print("tail gradient nonzero",bool(model.tail[0].bias.grad.abs().sum()>0))
print("initial gate gradient",float(model.gate[0].weight.grad.abs().max()))

## Repeat for RGB and four bands

The experiment has zero optimizer steps. It records shapes/counts and checks gradients; it does not produce accuracy or transfer evidence.

In [ ]:
from experiment import run
report=run()
for case in report["cases"]:
    print(case["bands"],case["parameters"],case["output_shape"],case["zero_residual_max_abs_bicubic_difference"])
print(report["limitations"])

## Exercise and interpretation

With RGB and scale2, the pre-shuffle tail has12 channels. With baseline.4, residual.1 and gate.5, output is.45. Since dy/dg=R, the initial zero residual explains zero gate gradients. Neither a larger output nor a large gate certifies physical detail. Matched-budget comparisons and valid-reference error crops remain open; historical single-scene results cannot close those requirements.